In [0]:
#Using the UDF in SQL queries
# You already know using UDFs with df using '.withColumn() 

In [0]:
from pyspark.sql import Row

# Employee performance data
employee_data = [
    Row(emp_id=101, emp_name="Alice Johnson", performance_score=92, years_service=5),
    Row(emp_id=102, emp_name="Bob Smith", performance_score=78, years_service=3),
    Row(emp_id=103, emp_name="Carol White", performance_score=55, years_service=2),
    Row(emp_id=104, emp_name="David Brown", performance_score=85, years_service=7),
    Row(emp_id=105, emp_name="Eve Davis", performance_score=95, years_service=4),
    Row(emp_id=106, emp_name="Frank Miller", performance_score=68, years_service=6),
]

employees_df = spark.createDataFrame(employee_data)
display(employees_df)

In [0]:
from pyspark.sql.functions import udf

@udf(returnType = "double")
def calculate_bonus(score, years):
    if score is None or years is None:
        return None
    
    base_bonus = score *10
    if years>=5:
        loyalty_bonus = base_bonus*1.2
        return loyalty_bonus
    else:
        return base_bonus



In [0]:
from pyspark.sql.functions import col
employees_with_rating = employees_df.withColumn("performance_bonus",
    calculate_bonus(
        col("performance_score"),
        col("years_service")) )

In [0]:
employees_df_bonus_calc.display()

In [0]:
'''
#ok exercise 3 : UDFs in SQL query

#Step 1: Register the UDF
spark.udf.register("function_name_in_sql", udf_function)

#Step 2 : create temporay sql view of the dataframe
df.createOrReplaceTempView("table_name")

#Step 3 : Use the function in SQL
SELECT * , function_name_in_sql(col1,col2)
FROM table_name

'''

In [0]:
#step 1 :
#register the UDF for SQL
spark.udf.register("calculate_bonus_sql", calculate_bonus)


#step 2 :
#create a temporary view
employees_with_rating.createOrReplaceTempView("employees_table")

In [0]:
sql_result = spark.sql(
""" 
SELECT *, 
calculate_bonus_sql(performance_score, years_service) AS bonus_amount
FROM employees_table
ORDER BY emp_id 
"""
)

In [0]:
sql_result.display()